# Imports

In [ ]:
import os
import sys
import copy
import time
import math
import hashlib
import json
from tqdm import tqdm
import pandas as pd
from pathlib import Path
import numpy as np
base = Path('/path/to/project')
scripts= base.joinpath('scripts')
sys.path.append(scripts.joinpath('pancancer_he_classifier'))
from helpers import anno as annoHelper
%load_ext autoreload
%autoreload 2
data = base.joinpath('/path/to/data')
raw=data.joinpath('liver','aofei_liver_annotation')

# 1) Liver: Sample dataframe, new template with fewer columns

In [ ]:
# img_pnfn = [x for x in raw.glob('*.svs')]
file_type = '.svs'
samples = pd.DataFrame([])
for i,im in enumerate(raw.glob('*%s' % file_type)):
    fn = im.parts[-1]
    geojson = [x for x in raw.glob('%s*.geojson' % fn.split('.')[0])]
    samples.loc[i,'slide'] = fn
    samples.loc[i,'type'] = 'Tumor'
    samples.loc[i,'slide_path'] = str(im)
   
    if len(geojson)==1:
        samples.loc[i,'annotation']= geojson[0].parts[-1]
        samples.loc[i,'anno_path'] = str(geojson[0])
    else:
        samples.loc[i,'annotation'] = np.nan
sampleinfo = raw.joinpath('sampleinfo')
sampleinfo.mkdir(exist_ok=True)
fn = sampleinfo.joinpath('samples_%d_slides.tsv' % samples.shape[0])
print(fn)
samples.to_csv(fn,
               sep='\t')
samples.head()

# 2) Skin (train)

In [ ]:
raw=data.joinpath('skin','data','HE')
anno = data.joinpath('skin','results','qupath','annotation_gjson')
file_type = '.svs'
samples = pd.DataFrame([])
for i,im in enumerate(raw.glob('*%s' % file_type)):
    fn = im.parts[-1]
    geojson = [x for x in anno.glob('%s*.geojson' % fn.split('.')[0])]
    samples.loc[i,'slide'] = fn
    samples.loc[i,'type'] = 'Tumor'
    samples.loc[i,'slide_path'] = str(im)
   
    if len(geojson)==1:
        samples.loc[i,'annotation']= geojson[0].parts[-1]
        samples.loc[i,'anno_path'] = str(geojson[0])
    else:
        samples.loc[i,'annotation'] = np.nan
sampleinfo = data.joinpath('skin','sampleinfo')
sampleinfo.mkdir(exist_ok=True)
fn = sampleinfo.joinpath('samples_%d_slides.tsv' % samples.shape[0])
print(fn)
samples.to_csv(fn,
               sep='\t')
min_samples = samples.loc[:,['slide','slide_path']].copy()
min_samples.loc[:,'task'] = np.array([x for x in range(1,min_samples.shape[0]+1)])
min_samples = min_samples.loc[:,['task','slide','slide_path']]
min_samples.to_csv(scripts.joinpath('pancancer_he_classifier','pipeline','samples.tsv'),
                   sep = '\t',      
                   index = False,
                  )
samples.loc[~samples.annotation.isna(),:]

In [ ]:
min_samples.head()

# 3) Acral skin (test)

In [ ]:
raw=data.joinpath('acral_melanoma_sarah')
# anno = data.joinpath('skin','results','qupath','annotation_gjson')
file_type = '.svs'
samples = pd.DataFrame([])
for i,im in enumerate(raw.glob('*%s' % file_type)):
    fn = im.parts[-1]
    geojson = [] #x for x in anno.glob('%s*.geojson' % fn.split('.')[0])]
    samples.loc[i,'task'] = int(i+1)
    samples.loc[i,'slide'] = fn
    samples.loc[i,'slide_path'] = str(im)
   
    if len(geojson)==1:
        samples.loc[i,'annotation']= geojson[0].parts[-1]
        samples.loc[i,'anno_path'] = str(geojson[0])
sampleinfo = base.joinpath('sampleinfo')
sampleinfo.mkdir(exist_ok=True)
fn = sampleinfo.joinpath('samples_%d_slides.tsv' % samples.shape[0])
samples.task = samples.task.astype(np.uint8)
print(fn)
samples.to_csv(fn,
               sep='\t',
               index = False)
samples.head()

In [ ]:
idx = samples.slide.str.contains('14732')
samples.loc[idx,:]